# Problem 8 - Lossless Data Compression | Executable Lab

**V1 fixed-length -> V2 RLE -> V3 min-heap Huffman -> V4 canonical Huffman -> V5 LZ77 + canonical Huffman**

**Group:** [fill in member names]  
**Assignment:** Problem 8, pages 31-34 of the official problem sheet.  
**Method:** implement both compression and decompression for every stage; prove exact recovery on the supplied datasets; compare compression efficiency and encoding/decoding throughput.

> The codec implementations are embedded below. The project folder includes 12 original datasets at three scales (`original/1x`, `original/5x`, `original/10x`), saved packets under matching `compressed/` folders, a `manifest.json`, and a single combined `compression_results.csv`. Run notebook cells from the project folder. No external benchmark utility module is required.


## Stage prerequisites: bit I/O, Huffman tree, canonical codes

The serializer stores enough header information to decode without relying on an external alphabet. All bitstreams are packed MSB first, with padding bits only in the final byte.

In [ ]:
from __future__ import annotations

from collections import Counter, defaultdict, deque
from dataclasses import dataclass
import heapq
import math
import struct
from typing import Optional


class BitWriter:
    def __init__(self):
        self.data = bytearray()
        self.accumulator = 0
        self.filled = 0
        self.bits_written = 0

    def write(self, value: int, width: int) -> None:
        if width < 0 or value < 0 or value >= (1 << width):
            raise ValueError(f"Value {value} does not fit in {width} bits")
        self.bits_written += width
        for bit_index in range(width - 1, -1, -1):
            self.accumulator = (self.accumulator << 1) | ((value >> bit_index) & 1)
            self.filled += 1
            if self.filled == 8:
                self.data.append(self.accumulator)
                self.accumulator = 0
                self.filled = 0

    def finish(self) -> bytes:
        if self.filled:
            self.data.append(self.accumulator << (8 - self.filled))
            self.accumulator = 0
            self.filled = 0
        return bytes(self.data)


class BitReader:
    def __init__(self, payload: bytes):
        self.data = payload
        self.pos = 0

    def read(self, width: int) -> int:
        if width < 0 or self.pos + width > 8 * len(self.data):
            raise ValueError("Truncated bitstream")
        value = 0
        for _ in range(width):
            byte = self.data[self.pos >> 3]
            value = (value << 1) | ((byte >> (7 - (self.pos & 7))) & 1)
            self.pos += 1
        return value


@dataclass(slots=True)
class HuffNode:
    weight: int
    symbol: Optional[int] = None
    left: Optional['HuffNode'] = None
    right: Optional['HuffNode'] = None


def huffman_tree(frequencies: dict[int, int]) -> Optional[HuffNode]:
    """Deterministic O(m log m) binary min-heap Huffman construction."""
    heap = []
    serial = 0
    for symbol, count in sorted(frequencies.items()):
        if count <= 0:
            raise ValueError("Frequencies must be positive")
        leaf = HuffNode(count, symbol)
        heapq.heappush(heap, (count, symbol, serial, leaf))
        serial += 1
    while len(heap) > 1:
        wa, mina, _, left = heapq.heappop(heap)
        wb, minb, _, right = heapq.heappop(heap)
        parent = HuffNode(wa + wb, None, left, right)
        heapq.heappush(heap, (wa + wb, min(mina, minb), serial, parent))
        serial += 1
    return heap[0][-1] if heap else None


def tree_codes(node: HuffNode) -> dict[int, tuple[int, int]]:
    codes = {}

    def visit(n: HuffNode, code: int, depth: int) -> None:
        if n.symbol is not None:
            codes[n.symbol] = (code, depth or 1)
        else:
            visit(n.left, code << 1, depth + 1)
            visit(n.right, (code << 1) | 1, depth + 1)

    visit(node, 0, 0)
    return codes


def canonical_codes(lengths: dict[int, int]) -> dict[int, tuple[int, int]]:
    """Sort (length, symbol); first code 0, others (code+1) << length jump."""
    if not lengths:
        return {}
    sorted_items = sorted(lengths.items(), key=lambda e: (e[1], e[0]))
    output = {}
    prior_length = sorted_items[0][1]
    if not 1 <= prior_length <= 255:
        raise ValueError("Invalid Huffman code length")
    code = 0
    for index, (symbol, length) in enumerate(sorted_items):
        if index:
            code = (code + 1) << (length - prior_length)
        if length <= 0 or length > 255 or code >= (1 << length):
            raise ValueError("Invalid / non-prefix-free code lengths")
        output[symbol] = (code, length)
        prior_length = length
    return output


def build_canonical(data: list[int] | bytes) -> dict[int, tuple[int, int]]:
    counts = Counter(data)
    if not counts:
        return {}
    root = huffman_tree(counts)
    lengths = {symbol: width for symbol, (_, width) in tree_codes(root).items()}
    return canonical_codes(lengths)


def serialize_lengths(codes: dict[int, tuple[int, int]], symbol_width: int) -> bytes:
    if len(codes) > 65535:
        raise ValueError("Too many symbols")
    result = bytearray(struct.pack('>H', len(codes)))
    for symbol in sorted(codes):
        result.extend(int(symbol).to_bytes(symbol_width, 'big'))
        result.append(codes[symbol][1])
    return bytes(result)


def parse_lengths(packet: bytes, cursor: int, symbol_width: int):
    if cursor + 2 > len(packet):
        raise ValueError("Truncated codebook")
    m = struct.unpack_from('>H', packet, cursor)[0]
    cursor += 2
    lengths = {}
    for _ in range(m):
        if cursor + symbol_width + 1 > len(packet):
            raise ValueError("Truncated codebook entries")
        symbol = int.from_bytes(packet[cursor:cursor + symbol_width], 'big')
        length = packet[cursor + symbol_width]
        cursor += symbol_width + 1
        if symbol in lengths:
            raise ValueError("Duplicate codebook symbol")
        lengths[symbol] = length
    return canonical_codes(lengths), cursor


def decode_one(reader: BitReader, codes: dict[int, tuple[int, int]]) -> int:
    if not codes:
        raise ValueError("Attempt to decode with empty table")
    decoding = {(width, bits): sym for sym, (bits, width) in codes.items()}
    maximum = max(width for _, width in codes.values())
    value = 0
    for width in range(1, maximum + 1):
        value = (value << 1) | reader.read(1)
        if (width, value) in decoding:
            return decoding[(width, value)]
    raise ValueError("Invalid Huffman codeword")


def decoder_for(codes: dict[int, tuple[int, int]]):
    """Precompute lookup once rather than per decoded symbol."""
    lookup = {(width, code): sym for sym, (code, width) in codes.items()}
    maximum = max((width for _, width in codes.values()), default=0)

    def get(reader: BitReader) -> int:
        value = 0
        for width in range(1, maximum + 1):
            value = (value << 1) | reader.read(1)
            if (width, value) in lookup:
                return lookup[width, value]
        raise ValueError("Invalid canonical bitstring")
    return get




## V1 — Fixed-length binary encoding and V2 — Run-length encoding

V1 assigns integer IDs to all observed byte values with $b=\lceil\log_2(m)\rceil$ bits (special-case $m=1$: zero-bit codewords are possible because the header stores the data length). V2 uses a 7-bit run length in $[1,127]$ followed by an 8-bit literal. **Theoretical Q1 RLE uses 3 bits per symbol rather than 8**, because that specific example has only six symbols.

In [ ]:
def v1_fixed_length_compress(data: bytes) -> bytes:
    alphabet = bytes(sorted(set(data)))
    m = len(alphabet)
    width = (m - 1).bit_length() if m else 0
    writer = BitWriter()
    index = {symbol: i for i, symbol in enumerate(alphabet)}
    for symbol in data:
        writer.write(index[symbol], width)
    return b'FL01' + struct.pack('>IHB', len(data), m, width) + alphabet + writer.finish()


def v1_fixed_length_decompress(packet: bytes) -> bytes:
    if len(packet) < 11 or packet[:4] != b'FL01':
        raise ValueError('Invalid fixed-length packet')
    n, m, width = struct.unpack_from('>IHB', packet, 4)
    if m > 256 or width != ((m - 1).bit_length() if m else 0) or (n > 0 and m == 0):
        raise ValueError('Invalid alphabet metadata')
    if len(packet) < 11 + m:
        raise ValueError('Truncated alphabet')
    alphabet = packet[11:11 + m]
    reader = BitReader(packet[11 + m:])
    result = bytearray()
    for _ in range(n):
        index = reader.read(width)
        if index >= m:
            raise ValueError('Invalid symbol index')
        result.append(alphabet[index])
    return bytes(result)


def v2_rle_compress(data: bytes) -> bytes:
    """7-bit run count (1..127) + 8-bit byte literal, packed into 15-bit runs.

    Handworked Part A uses 3-bit symbols because its alphabet has only 6 letters.
    """
    writer = BitWriter()
    i = 0
    while i < len(data):
        count = 1
        while i + count < len(data) and count < 127 and data[i + count] == data[i]:
            count += 1
        writer.write(count, 7)
        writer.write(data[i], 8)
        i += count
    return b'RL02' + struct.pack('>I', len(data)) + writer.finish()


def v2_rle_decompress(packet: bytes) -> bytes:
    if len(packet) < 8 or packet[:4] != b'RL02':
        raise ValueError('Invalid RLE packet')
    n = struct.unpack_from('>I', packet, 4)[0]
    reader = BitReader(packet[8:])
    decoded = bytearray()
    while len(decoded) < n:
        count, symbol = reader.read(7), reader.read(8)
        if count == 0 or len(decoded) + count > n:
            raise ValueError('Invalid RLE run')
        decoded.extend(bytes([symbol]) * count)
    return bytes(decoded)


def write_tree(node: HuffNode, writer: BitWriter) -> None:
    if node.symbol is not None:
        writer.write(1, 1)
        writer.write(node.symbol, 8)
    else:
        writer.write(0, 1)
        write_tree(node.left, writer)
        write_tree(node.right, writer)


def read_tree(reader: BitReader) -> HuffNode:
    if reader.read(1):
        return HuffNode(0, reader.read(8))
    left = read_tree(reader)
    right = read_tree(reader)
    return HuffNode(0, None, left, right)




## V3 — Tree-based Huffman and V4 — Canonical Huffman

V3 builds a Huffman tree with `heapq`, writes the tree in preorder, then writes variable-length payload codewords. V4 reconstructs codewords using **lengths and a known ordered alphabet**. In this standalone file format, it serializes each `(symbol, length)` pair and therefore does **not** always outperform a compact explicit Huffman tree header. Both codecs are compared including total metadata. Both handle the empty and one-symbol cases.

In [ ]:
def v3_huffman_tree_compress(data: bytes) -> bytes:
    writer = BitWriter()
    if data:
        root = huffman_tree(Counter(data))
        write_tree(root, writer)
        codes = tree_codes(root)
        for symbol in data:
            code, width = codes[symbol]
            writer.write(code, width)
    return b'HF03' + struct.pack('>I', len(data)) + writer.finish()


def v3_huffman_tree_decompress(packet: bytes) -> bytes:
    if len(packet) < 8 or packet[:4] != b'HF03':
        raise ValueError('Invalid Huffman tree packet')
    n = struct.unpack_from('>I', packet, 4)[0]
    if n == 0:
        return b''
    reader = BitReader(packet[8:])
    root = read_tree(reader)
    result = bytearray()
    for _ in range(n):
        node = root
        if node.symbol is not None:
            if reader.read(1) != 0:
                raise ValueError('Invalid single-symbol code')
        else:
            while node.symbol is None:
                node = node.right if reader.read(1) else node.left
        result.append(node.symbol)
    return bytes(result)


def v4_huffman_canonical_compress(data: bytes) -> bytes:
    codes = build_canonical(data)
    writer = BitWriter()
    for symbol in data:
        bits, width = codes[symbol]
        writer.write(bits, width)
    return (b'HC04' + struct.pack('>I', len(data)) +
            serialize_lengths(codes, 1) + writer.finish())


def v4_huffman_canonical_decompress(packet: bytes) -> bytes:
    if len(packet) < 8 or packet[:4] != b'HC04':
        raise ValueError('Invalid canonical Huffman packet')
    n = struct.unpack_from('>I', packet, 4)[0]
    codes, cursor = parse_lengths(packet, 8, 1)
    read_symbol = decoder_for(codes)
    reader = BitReader(packet[cursor:])
    return bytes(read_symbol(reader) for _ in range(n))




## V5 — LZ77 (W=4096, L=32) + three canonical Huffman tables

The encoder finds the longest valid backward match and emits `(offset, length, next_literal)`. It reserves one final literal for every token. `length=0` means a standalone literal. Huffman coding uses separate canonical tables for length values, nonzero offsets, and next-literal bytes. This is an **educational chained LZ77–canonical-Huffman codec**, not a bit-compatible implementation of the standardized DEFLATE format. Its self-contained packet stores the three sparse codebooks.

In [ ]:
def lz77_tokens(data: bytes, window: int = 4096, lookahead: int = 32):
    """Longest-match LZ77 triples (offset,length,next_literal).

    Full exact search of matching three-byte-prefix candidate positions in the
    previous W bytes; matching from 1 or 2 bytes uses last seen positions.
    Supports overlapping back-references and consumes a final literal always.
    """
    if window < 1 or lookahead < 1:
        raise ValueError('Window and lookahead must be positive')
    n = len(data)
    p = 0
    indexed_until = 0
    expired_until = 0
    trigrams = defaultdict(deque)
    last_one = {}
    last_two = {}
    output = []
    while p < n:
        # Add all positions processed by preceding token, even if referenced.
        while indexed_until < p:
            j = indexed_until
            last_one[data[j]] = j
            if j + 2 <= n:
                last_two[data[j:j+2]] = j
            if j + 3 <= n:
                trigrams[data[j:j+3]].append(j)
            indexed_until += 1
        # Maintain only positions starting in S[max(0,p-W)..p-1].
        oldest_valid = max(0, p-window)
        while expired_until < oldest_valid:
            j = expired_until
            if j + 3 <= n:
                key = data[j:j+3]
                positions = trigrams.get(key)
                if positions and positions[0] == j:
                    positions.popleft()
                    if not positions:
                        del trigrams[key]
            expired_until += 1
        maximum = min(lookahead, n-p-1)  # reserve a final next_char
        best_len = 0
        best_off = 0
        if maximum >= 3:
            for j in reversed(trigrams.get(data[p:p+3], ())):
                if j < oldest_valid:
                    break
                length = 3
                while length < maximum and data[j+length] == data[p+length]:
                    length += 1
                if length > best_len:
                    best_len, best_off = length, p-j
                if best_len == maximum:
                    break
        if best_len < 2 and maximum >= 2:
            j = last_two.get(data[p:p+2], -1)
            if oldest_valid <= j < p:
                best_len, best_off = 2, p-j
        if best_len < 1 and maximum >= 1:
            j = last_one.get(data[p], -1)
            if oldest_valid <= j < p:
                best_len, best_off = 1, p-j
        output.append((best_off, best_len, data[p+best_len]))
        p += best_len + 1
    return output


def v5_lz77_deflate_compress(data: bytes) -> bytes:
    tokens = lz77_tokens(data, window=4096, lookahead=32)
    # Three independently canonical-Huffman-coded streams. A length of zero
    # designates a literal; a positive length is followed by a coded offset.
    lengths = [length for _, length, _ in tokens]
    offsets = [offset for offset, length, _ in tokens if length]
    literals = [literal for _, _, literal in tokens]
    length_codes = build_canonical(lengths)
    offset_codes = build_canonical(offsets)
    literal_codes = build_canonical(literals)
    writer = BitWriter()
    for offset, length, literal in tokens:
        writer.write(*length_codes[length])
        if length:
            writer.write(*offset_codes[offset])
        writer.write(*literal_codes[literal])
    header = (b'LZ05' + struct.pack('>II', len(data), len(tokens)) +
              serialize_lengths(length_codes, 1) +
              serialize_lengths(offset_codes, 2) +
              serialize_lengths(literal_codes, 1))
    return header + writer.finish()


def v5_lz77_deflate_decompress(packet: bytes) -> bytes:
    if len(packet) < 12 or packet[:4] != b'LZ05':
        raise ValueError('Invalid LZ77+Huffman packet')
    n, token_count = struct.unpack_from('>II', packet, 4)
    length_codes, cursor = parse_lengths(packet, 12, 1)
    offset_codes, cursor = parse_lengths(packet, cursor, 2)
    literal_codes, cursor = parse_lengths(packet, cursor, 1)
    get_length = decoder_for(length_codes)
    get_offset = decoder_for(offset_codes)
    get_literal = decoder_for(literal_codes)
    reader = BitReader(packet[cursor:])
    result = bytearray()
    for _ in range(token_count):
        length = get_length(reader)
        if length:
            offset = get_offset(reader)
            if not 1 <= length <= 32 or not 1 <= offset <= min(4096, len(result)):
                raise ValueError('Invalid LZ77 reference')
            for _ in range(length):
                result.append(result[-offset])  # overlap is intentionally legal
        result.append(get_literal(reader))
        if len(result) > n:
            raise ValueError('Output exceeds declared uncompressed size')
    if len(result) != n:
        raise ValueError('Incorrect LZ77 output size')
    return bytes(result)




In [ ]:
CODECS = {
    'V1 Fixed-length': (v1_fixed_length_compress, v1_fixed_length_decompress),
    'V2 RLE': (v2_rle_compress, v2_rle_decompress),
    'V3 Huffman tree': (v3_huffman_tree_compress, v3_huffman_tree_decompress),
    'V4 Canonical Huffman': (v4_huffman_canonical_compress, v4_huffman_canonical_decompress),
    'V5 LZ77 + Canonical': (v5_lz77_deflate_compress, v5_lz77_deflate_decompress),
}



## Validate the theoretical hand traces (Q1–Q4)

These executable checks corroborate manual work; they **do not substitute for handwritten traces**.

In [ ]:
import math

counts = {'A':45, 'B':13, 'C':12, 'D':16, 'E':9, 'F':5}
h = -sum((x/100)*math.log2(x/100) for x in counts.values())
assert abs(h - 2.219879978107314) < 1e-12
assert math.ceil(math.log2(6))*100 == 300
assert 6*10 == 60 and 100*10 == 1000

original_codes = {'A':'0', 'B':'101', 'C':'100', 'D':'111', 'E':'1101', 'F':'1100'}
assert sum(counts[c]*len(code) for c,code in original_codes.items()) == 224
canonical_expected = {'A':'0', 'B':'100', 'C':'101', 'D':'110', 'E':'1110', 'F':'1111'}
canonical_result = canonical_codes({ord(s):len(code) for s,code in original_codes.items()})
assert {chr(s):format(bits,f'0{width}b') for s,(bits,width) in canonical_result.items()} == canonical_expected
bitstream='0101011001110'
reverse={code:s for s,code in original_codes.items()}
decoded='';partial=''
for bit in bitstream:
    partial+=bit
    if partial in reverse:
        decoded+=reverse[partial];partial=''
assert decoded=='ABAFDA' and partial==''
trace=lz77_tokens(b'ABRACADABRA ABRACADABRA',window=12,lookahead=12)
expected=[(0,0,ord('A')),(0,0,ord('B')),(0,0,ord('R')),(3,1,ord('C')),(2,1,ord('D')),(7,4,ord(' ')),(12,10,ord('A'))]
assert trace==expected and trace[5][2] == 32  # ASCII space, never underscore
print(f'Q1 entropy: {h:.6f} bits/symbol; bound: {100*h:.3f} bits; Huffman payload: 224 bits')
print('Q2 decoded 13 bits:', decoded)
print('Q3 canonical codes:',canonical_expected)
print('Q4 7 triples:',[(o,l,chr(c)) for o,l,c in trace])

Q1 entropy: 2.219880 bits/symbol; bound: 221.988 bits; Huffman payload: 224 bits
Q2 decoded 13 bits: ABAFDA
Q3 canonical codes: {'A': '0', 'B': '100', 'C': '101', 'D': '110', 'E': '1110', 'F': '1111'}
Q4 7 triples: [(0, 0, 'A'), (0, 0, 'B'), (0, 0, 'R'), (3, 1, 'C'), (2, 1, 'D'), (7, 4, ' '), (12, 10, 'A')]


## Extra edge cases (empty messages, all 256 byte values, overlap)

Round-trip verification uses `assert decompress(compress(data)) == data`. The reverse LZ77 copy is performed one byte at a time to support overlapping references.

In [ ]:
edge_samples = [b'', b'A', b'AAAA', b'ABABABAB', b'A'*5000, bytes(range(256)), bytes(range(256))*2, b'ABRACADABRA ABRACADABRA', bytes([0,255])*100]
for name,(enc,dec) in CODECS.items():
    for original in edge_samples:
        assert dec(enc(original)) == original, (name,len(original))
print(f'Edge-case round trips passed: {len(CODECS)*len(edge_samples)} / {len(CODECS)*len(edge_samples)}')

Edge-case round trips passed: 45 / 45


## Supplied datasets and combined benchmark results

The package contains four dataset types at each of three scales: Standard (1x), 5x, and 10x. The originals are in `original/{1x,5x,10x}/`; compressed packets are in `compressed/{1x,5x,10x}/{codec_id}/`. `manifest.json` records expected byte counts and SHA-256 hashes for every original and packet.

`compression_results.csv` combines the per-scale results into one table with 60 rows (12 datasets, 5 codecs each). It records packet size, saved percentage, checksums, round-trip status, and the benchmark's encode/decode times. The timing values are a recorded snapshot and vary across machines and runs.


In [ ]:
from pathlib import Path
import hashlib, json
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
assert (ROOT / "manifest.json").is_file(), "Run this notebook from the project folder."
manifest = json.loads((ROOT / "manifest.json").read_text(encoding="utf-8"))

CODEC_LABELS = {
    "v1_fixed_length": "V1 Fixed-length",
    "v2_rle": "V2 RLE",
    "v3_huffman_tree": "V3 Huffman tree",
    "v4_huffman_canonical": "V4 Canonical Huffman",
    "v5_lz77_canonical": "V5 LZ77 + Canonical",
}

def manifest_path(relative_path, scale):
    relative_path = Path(relative_path)
    scale_dir = {"Standard": "1x"}.get(scale, scale)
    return ROOT / relative_path.parts[0] / scale_dir / Path(*relative_path.parts[1:])

# Check every original and saved packet against the manifest, then decode it.
checked_packets = 0
for dataset in manifest["datasets"]:
    original = manifest_path(dataset["original_file"], dataset["scale"]).read_bytes()
    assert len(original) == dataset["original_bytes"]
    assert hashlib.sha256(original).hexdigest() == dataset["original_sha256"]
    for packet_info in dataset["packets"]:
        packet = manifest_path(packet_info["file"], dataset["scale"]).read_bytes()
        assert len(packet) == packet_info["packet_bytes"]
        assert hashlib.sha256(packet).hexdigest() == packet_info["packet_sha256"]
        decoder = CODECS[CODEC_LABELS[packet_info["codec_id"]]][1]
        assert decoder(packet) == original, (dataset["name"], packet_info["codec_id"])
        checked_packets += 1

results = pd.read_csv(ROOT / "compression_results.csv")
assert len(manifest["datasets"]) == 12
assert len(results) == checked_packets == 60
assert results["roundtrip"].eq("PASS").all()
print(f"PASS: {checked_packets}/60 packets match their recorded hashes and decompress byte-for-byte.")


## Compare packet sizes and recorded throughput

The next cell joins the combined CSV to the dataset scales, converts its recorded encode/decode durations to MiB/s, and displays the measurements. Packet sizes include headers and padding. Negative `saved_pct` values mean the packet is larger than its input. Throughput is a machine-specific snapshot, not a controlled cross-machine comparison.


In [ ]:
scale_by_dataset = {dataset["name"]: dataset["scale"] for dataset in manifest["datasets"]}
results["scale"] = results["dataset"].map(scale_by_dataset)
results["codec"] = results["codec_id"].map(CODEC_LABELS)
results["encode_MiBs"] = results["original_bytes"] / (1024 ** 2) / results["encode_seconds"]
results["decode_MiBs"] = results["original_bytes"] / (1024 ** 2) / results["decode_seconds"]

measurements = results[[
    "scale", "dataset", "codec", "packet_bytes", "saved_pct",
    "encode_MiBs", "decode_MiBs", "roundtrip",
]].copy()
display(measurements.round({"saved_pct": 2, "encode_MiBs": 2, "decode_MiBs": 2}))


## Visualize compression and throughput by scale

These figures are generated from the combined CSV. One panel per scale keeps the 12 datasets readable; the zero line separates size savings from expansion.


In [ ]:
import matplotlib.pyplot as plt

scale_order = ["Standard", "5x", "10x"]
fig, axes = plt.subplots(1, 3, figsize=(19, 6), sharey=True)
for ax, scale in zip(axes, scale_order):
    subset = results[results["scale"] == scale]
    chart = subset.pivot(index="dataset", columns="codec", values="saved_pct")
    chart = chart[[CODEC_LABELS[key] for key in CODEC_LABELS]]
    chart.plot(kind="barh", ax=ax, width=0.82)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title(f"{scale} scale")
    ax.set_xlabel("Bytes saved (%)")
    ax.set_ylabel("")
    ax.grid(axis="x", alpha=0.25)
axes[-1].legend(title="Codec", bbox_to_anchor=(1.02, 1), loc="upper left")
fig.suptitle("Packet size change by dataset and codec")
fig.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(19, 6), sharey=True)
for ax, scale in zip(axes, scale_order):
    subset = results[results["scale"] == scale]
    chart = subset.pivot(index="dataset", columns="codec", values="encode_MiBs")
    chart = chart[[CODEC_LABELS[key] for key in CODEC_LABELS]]
    chart.plot(kind="barh", ax=ax, width=0.82)
    ax.set_title(f"{scale} scale")
    ax.set_xlabel("Encode throughput (MiB/s)")
    ax.set_ylabel("")
    ax.grid(axis="x", alpha=0.25)
axes[-1].legend(title="Codec", bbox_to_anchor=(1.02, 1), loc="upper left")
fig.suptitle("Recorded encode throughput by dataset and codec")
fig.tight_layout()
plt.show()


In [ ]:
## Summarize the results

for dataset_name, subset in results.groupby("dataset", sort=False):
    best = subset.loc[subset["packet_bytes"].idxmin()]
    print(f"{dataset_name}: smallest packet = {best['codec']}, {best['packet_bytes']:,} bytes ({best['saved_pct']:+.2f}% saved)")

all_symbols = bytes(range(256)) * 20
v3_size = len(CODECS["V3 Huffman tree"][0](all_symbols))
v4_size = len(CODECS["V4 Canonical Huffman"][0](all_symbols))
assert v4_size > v3_size
print(f"Canonical header counterexample (256 distinct symbols): V3={v3_size} B; V4={v4_size} B")


## Interpret results and limitations

- The supplied manifest covers **12 datasets** (four data types at 1x, 5x, and 10x); the combined results contain **60 codec/dataset rows**. The verification cell checks each original and packet hash and confirms byte-perfect decompression.
- No codec wins for every dataset: RLE is effective on long identical runs; Huffman coding benefits from skewed byte frequencies; LZ77 can exploit repeated phrases. Random bytes are generally incompressible, so packet headers and coding metadata can make the output larger.
- V5 is **not RFC 1951/DEFLATE compatible**. It is an educational LZ77 plus canonical-Huffman codec with a custom packet header and three codebooks. It codes short matches too, which can increase packet size on some inputs.
- The LZ77 token index retains data proportional to input size while building canonical tables, so its encoder uses additional memory for large inputs.
- The encode/decode times in `compression_results.csv` are recorded measurements; hardware, Python version, and system load affect throughput. Size and checksum results are reproducible for the supplied files.
